In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error

In [6]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [8]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [10]:
tahminler = []
tscv = TimeSeriesSplit(n_splits=2)  # Daha güvenli split
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        # Train: 2002-2021, Validation: 2022, Test: 2023
        train_data = df_target[df_target['yıl'] <= 2021]
        val_data = df_target[df_target['yıl'] == 2022]
        test_data = df_target[df_target['yıl'] == 2023]

        if val_data.empty or test_data.empty:
            continue

        X_train = train_data[features]
        y_train = train_data[hedef]
        X_val = val_data[features]
        y_val = val_data[hedef]
        X_test = test_data[features]
        y_test = test_data[hedef]

        # Küçük veya sabit veri seti kontrolü
        if len(X_train) < 10 or len(set(y_train)) <= 1:
            continue

        # Linear Regression (sabit)
        lr = LinearRegression().fit(X_train, y_train)

        # Random Forest - GridSearchCV (korumalı)
        try:
            rf_params = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
            rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=tscv,
                                   n_jobs=-1, scoring=mae_scorer)
            rf_grid.fit(X_train, y_train)
            rf = rf_grid.best_estimator_
        except:
            rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

        # XGBoost (sabit)
        xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)

        # SVR - RandomizedSearchCV (korumalı)
        try:
            svr_params = {'C': [1, 10], 'epsilon': [0.1]}
            svr_rand = RandomizedSearchCV(SVR(), svr_params, cv=tscv, n_iter=2, n_jobs=-1,
                                          random_state=42, scoring=mae_scorer)
            svr_rand.fit(X_train, y_train)
            svr = svr_rand.best_estimator_
        except:
            svr = SVR(C=1.0, epsilon=0.1).fit(X_train, y_train)

        # KNN - GridSearchCV (korumalı, dinamik komşu ayarı)
        try:
            max_neighbors = min(7, len(X_train))
            knn_k_list = [k for k in [3, 5, 7] if k <= max_neighbors]

            if knn_k_list:
                knn_params = {'n_neighbors': knn_k_list}
                knn_grid = GridSearchCV(KNeighborsRegressor(), knn_params, cv=tscv,
                                        n_jobs=-1, scoring=mae_scorer)
                knn_grid.fit(X_train, y_train)
                knn = knn_grid.best_estimator_
            else:
                knn = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)
        except:
            knn = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)

        # Tahmin ve skorlar
        y_val_pred_lr = lr.predict(X_val)[0]
        y_val_pred_rf = rf.predict(X_val)[0]
        y_val_pred_xgb = xgb.predict(X_val)[0]
        y_val_pred_svr = svr.predict(X_val)[0]
        y_val_pred_knn = knn.predict(X_val)[0]

        y_test_pred_lr = lr.predict(X_test)[0]
        y_test_pred_rf = rf.predict(X_test)[0]
        y_test_pred_xgb = xgb.predict(X_test)[0]
        y_test_pred_svr = svr.predict(X_test)[0]
        y_test_pred_knn = knn.predict(X_test)[0]

        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2022": y_val.values[0],
            "tahmin_2022_lr": y_val_pred_lr,
            "tahmin_2022_rf": y_val_pred_rf,
            "tahmin_2022_xgb": y_val_pred_xgb,
            "tahmin_2022_svr": y_val_pred_svr,
            "tahmin_2022_knn": y_val_pred_knn,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": y_test_pred_lr,
            "tahmin_2023_rf": y_test_pred_rf,
            "tahmin_2023_xgb": y_test_pred_xgb,
            "tahmin_2023_svr": y_test_pred_svr,
            "tahmin_2023_knn": y_test_pred_knn,
        })


In [12]:
df_tahmin = pd.DataFrame(tahminler)

In [14]:
# Doğruluk oranı hesaplama (2023 yılı için)
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    tahmin_kolonu = f"tahmin_2023_{model}"
    df_tahmin[f"{model}_yuzde_dog"] = 100 - (
        (np.abs(df_tahmin["gercek_2023"] - df_tahmin[tahmin_kolonu]) / df_tahmin["gercek_2023"]) * 100
    )
    # Değerleri 0–100 aralığına kısıtla
    df_tahmin[f"{model}_yuzde_dog"] = df_tahmin[f"{model}_yuzde_dog"].clip(lower=0, upper=100)
print("\n📊 Şehir Bazlı Ortalama Doğruluklar (2023):")
print("-" * 40)
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = df_tahmin[f"{model}_yuzde_dog"].mean()
    print(f" {model.upper():<4} doğruluk ort.: {ortalama:.2f}%")
# Tahmin detayları içeren DataFrame'i CSV'ye yaz
df_tahmin.to_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv", index=False)
print("\n✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_5model.csv' olarak kaydedildi.")


📊 Şehir Bazlı Ortalama Doğruluklar (2023):
----------------------------------------
 LR   doğruluk ort.: 67.85%
 RF   doğruluk ort.: 78.87%
 XGB  doğruluk ort.: 79.89%
 SVR  doğruluk ort.: 70.43%
 KNN  doğruluk ort.: 79.18%

✅ Tahmin doğrulukları 'sehir_bazli_2023_tahmin_dogruluk_5model.csv' olarak kaydedildi.


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\n📉 Model Performans Metrikleri (2023):")
print("{:<6} | {:>10} | {:>10}".format("Model", "MAE", "RMSE"))
print("-" * 32)

metric_sonuclar = []

for model in ["lr", "rf", "xgb", "svr", "knn"]:
    y_true = df_tahmin["gercek_2023"]
    y_pred = df_tahmin[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)

    print("{:<6} | {:>10.2f} | {:>10.2f}".format(model.upper(), mae, rmse))

    metric_sonuclar.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)
    })
# Metrikleri CSV olarak kaydet
df_metrics = pd.DataFrame(metric_sonuclar)
df_metrics.to_csv("sehir_bazli_2023_model_metrikleri.csv", index=False)
print("\n✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.")


📉 Model Performans Metrikleri (2023):
Model  |        MAE |       RMSE
--------------------------------
LR     |     385.10 |    1676.15
RF     |     297.69 |    1052.59
XGB    |     248.50 |     880.46
SVR    |     524.30 |    1938.14
KNN    |     296.30 |     982.87

✅ Performans metrikleri 'sehir_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared err

In [18]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Bölge ekle
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre grupla ve toplamları al
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[[
    "gercek_2023",
    "tahmin_2023_lr",
    "tahmin_2023_rf",
    "tahmin_2023_xgb",
    "tahmin_2023_svr",
    "tahmin_2023_knn"
]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin5model.csv", index=False)

In [20]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv")  # Gerekirse yolunu düzenle

# 2. Doğruluk hesaplama fonksiyonu
def hesapla_dog(model_etiketi):
    df[f"{model_etiketi}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model_etiketi}"]) / df["gercek_2023"] * 100
    )
    df[f"{model_etiketi}_yuzde_dog"] = df[f"{model_etiketi}_yuzde_dog"].clip(0, 100)

# 3. Altı model için hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    hesapla_dog(model)

# 4. Ortalamaları yazdır
print("\nŞehir Bazlı Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 5. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler.csv", index=False)


Şehir Bazlı Doğruluk Ortalamaları (2023):
 - LR: 67.85%
 - RF: 78.87%
 - XGB: 79.89%
 - SVR: 70.43%
 - KNN: 79.18%


In [22]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin5model.csv")  # Gerekirse dosya adını değiştir

# 2. Her model için yüzdelik doğruluk hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model}"]) / df["gercek_2023"] * 100
    )
    df[f"{model}_yuzde_dog"] = df[f"{model}_yuzde_dog"].clip(0, 100)

# 3. Ortalamaları hesapla ve yazdır
print(f"\nBölge Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler.csv", index=False)


Bölge Doğruluk Ortalamaları (2023):
 - LR: 83.93%
 - RF: 84.88%
 - XGB: 87.42%
 - SVR: 73.97%
 - KNN: 84.89%


In [28]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin5model.csv")  # Dosya adını gerektiği gibi değiştir

# 2. Hata metriklerini hesapla
print("\n📉 Bölge Bazlı Model Performans Metrikleri (2023):")
print("{:<6} | {:>10} | {:>10} | {:>6}".format("Model", "MAE", "RMSE", "R²"))
print("-" * 45)

metric_sonuclar_bolge = []

for model in ["lr", "rf", "xgb", "svr", "knn"]:
    y_true = df["gercek_2023"]
    y_pred = df[f"tahmin_2023_{model}"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)

    print("{:<6} | {:>10.2f} | {:>10.2f} | {:>6.3f}".format(model.upper(), mae, rmse, r2))

    metric_sonuclar_bolge.append({
        "Model": model.upper(),
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 3)
    })

# 3. Sonuçları kaydet
df_metrics_bolge = pd.DataFrame(metric_sonuclar_bolge)
df_metrics_bolge.to_csv("bolge_bazli_2023_model_metrikleri.csv", index=False)
print("\n✅ Metrikler 'bolge_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.")



📉 Bölge Bazlı Model Performans Metrikleri (2023):
Model  |        MAE |       RMSE |     R²
---------------------------------------------
LR     |    3170.61 |    5824.08 |  0.887
RF     |    3142.82 |    5604.91 |  0.895
XGB    |    2525.59 |    4467.64 |  0.934
SVR    |    5850.07 |   10751.05 |  0.615
KNN    |    3037.67 |    5188.31 |  0.910

✅ Metrikler 'bolge_bazli_2023_model_metrikleri.csv' dosyasına kaydedildi.


C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared err

In [50]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)
shap.summary_plot(shap_values, X_train, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.savefig("xgb_shap_summary.png")
plt.close()


NotFittedError: need to call fit or load_model beforehand

In [46]:
from lime.lime_tabular import LimeTabularExplainer

explainer_lime = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    mode='regression',
    verbose=True
)

instance = X_train.iloc[0].values
lime_exp = explainer_lime.explain_instance(instance, xgb.predict)
lime_exp.save_to_file(f"lime_{il}_{hedef}.html")


AttributeError: module 'xgboost' has no attribute 'predict'